# meeting_extracted_phase0_explore — 會議紀錄萃取 Phase 0 探勘（一次性）

- 用途：回答進階規劃 v0.4.1 §11 Phase 0 的統計題，決定 T1 清洗規則與門檻。**只讀、不寫任何表。** 用完即封存，不升級成 Job（§2.3）。
- 輸入：`{catalog}.{schema}.{source_table}`（預設 `b_internal_meeting_record`）、`g_sales_brief_brand_summary` / `g_sales_brief_app_summary` / `g_sales_brief_market_intelligence`、`mi3_semantic.*`、`information_schema.tables`
- 輸出：stdout；最後 [c30] 印一段 JSON（只有統計、網域、AUO 部門代碼，**不含姓名、email**），整段貼回給 Agent
- 參數（widgets）：`catalog`、`schema`、`source_table`、`date_from` / `date_to`（對 `mail_date`）、`sb_periods`
- 用法：Run All → 複製 [c30] 輸出。serverless / all-purpose / job cluster 皆可；不用 cache。
- 負責人 / 更新日期：（填）/ 2026-10-14

| cell | 回答的問題（規劃章節） |
|---|---|
| [c11] | 筆數、`is_test` 比例、`create_date` vs `mail_date` 落差（§2.1 水位） |
| [c12] | 三個正文欄非空率、長度 P50/P95、oversize（§4.0 關卡：md 非空 < 80% 先停） |
| [c13] | `mail_from` 格式一致率、部門代碼、主旨 `[BU]` 標頭、單位判定衝突（§4.1b、§6.8、待決 #12/#13） |
| [c14] | md 標頭標記覆蓋率、`Sent:` 日期可解析率（§4.1 (c) 分段） |
| [c15] | 三正文一致性、`body_conflict` 比例（§4.1 (a) 4，待決 #1；> 10% 先停） |
| [c16] | 粗估 content_type 分布、每封段數（§4.1 (d)） |
| [c17] | email 網域、客戶出現頻率（§6.3 `entity_alias` 種子） |
| [c18] | 跨信重複粗估（§4.1 (e)；預期 30–60%） |
| [c19] | md 正文組成：base64、HTML、表格各佔多少（md 比其他正文大很多的原因） |
| [c20] | Sales Brief 三張文字表欄位、品牌實際值、填報時間（待決 #3、#10） |
| [c21] | `mi3_semantic.entity_alias` 是否存在、部門對照表候選（待決 #13、#15） |

粗估說明：content_type 與重複率是 **mail 層**近似（還沒有真正的分段與 token 化），只用來定門檻量級，不是 T1-b 的結果。

In [ ]:
# [c01] params
# 探勘用：只讀。widget 建立過後改預設值不會生效；要重設先 dbutils.widgets.removeAll()。
from datetime import date, datetime

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("source_table", "b_internal_meeting_record")
dbutils.widgets.text("date_from", "2026-06-01")  # 對 mail_date，含
dbutils.widgets.text("date_to", "2026-09-30")    # 對 mail_date，含
dbutils.widgets.text("sb_periods", "6")          # Sales Brief 看最近幾期


def _date(s: str) -> date:
    return datetime.strptime(s.strip(), "%Y-%m-%d").date()


cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "source_table": dbutils.widgets.get("source_table").strip(),
    "date_from": _date(dbutils.widgets.get("date_from")),
    "date_to": _date(dbutils.widgets.get("date_to")),
    "sb_periods": int(dbutils.widgets.get("sb_periods")),
}
assert cfg["catalog"] and cfg["schema"] and cfg["source_table"], "catalog / schema / source_table 不可為空"
assert cfg["date_from"] <= cfg["date_to"], "date_from 不可晚於 date_to"
print(cfg)

In [ ]:
# [c02] imports
# 本 cell 與 [c03]～[c06] 不碰 spark / dbutils，可被 tests/ 載入。
import json

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

BODY_FIELDS = ["mail_body_md", "mail_without_fw", "mail_body"]
SOURCE_COLS = [
    "pk_seq", "drupal_id", "create_date", "mail_date", "mail_datetime",
    "mail_from", "mail_to", "mail_cc", "mail_bcc", "mail_subject",
    *BODY_FIELDS, "is_fw", "has_attach", "is_test",
]  # 不讀 mail_body_html（base64 圖片，§4.0）

OVERSIZE_CHARS = 200_000  # §4.0
MIN_BODY_CHARS = 150      # §4.1 (a) 1 的 md 門檻，也是 noise 門檻
SHORT_CHARS = 300         # attachment_only / appointment 門檻
MINUTES_CHARS = 800

# 以下皆為 Java regex（Spark 用）
MAIL_FROM_RE = r"^(\d{6,8})/([^/]+)/([^/]+)/([A-Z0-9]+)$"
SUBJECT_PREFIX_RE = r"(?i)^(\s*(fwd|fw|re|回覆|轉寄)\s*([:：]|\s)\s*)+"  # 前綴後要有冒號或空白，避免吃到 Review
UNIT_TAG_RE = r"\[([A-Za-z]{2,5})\]"
IMG_RE = r"!\[[^\]]*\]\(cid:[^)]*\)"
WORD_JUNK_RE = r"(?m)^\s*(Normal|DocumentEmail|MicrosoftInternetExplorer4|false|true|EN-US|ZH-TW|X-NONE|\d{1,3})\s*$"
ENTITY_RE = r"&(nbsp|amp|lt|gt|quot|#\d+);"
MD_MARKERS = {
    "md_from": r"\*\*From:\*\*",
    "md_sent": r"\*\*Sent:\*\*",
    "md_to": r"\*\*To:\*\*",
    "md_cc": r"\*\*Cc:\*\*",
    "md_subject": r"\*\*Subject:\*\*",
    "md_when": r"\*\*When:\*\*",
    "md_where": r"\*\*Where:\*\*",
    "orig_sep": r"-----\s*Original (Message|Appointment)\s*-----",
    "plain_from": r"(?m)^\s*From:",
    "img": IMG_RE,
}
HEADER_LINE_RE = (
    r"\*\*(From|Sent|To|Cc|Bcc|Subject|When|Where):\*\*.*?(?=\*\*[A-Za-z]+:\*\*|\n|$)"
    r"|(?m)^\s*(From|Sent|To|Cc|Bcc|Subject|When|Where):.*$"
)
DISCLAIMER_RE = r"(?i)CONFIDENTIAL|保密|This email contains information|如您並非|has pledged to source"
ATTACH_KW_RE = r"(?i)see attached|如附件|file server|please refer the attached"
ITEM_HEADER_RE = r"(?m)^\s*ITEM\b"
MINUTES_RE = r"(?i)meeting minutes|會議紀要|會議記錄|會議紀錄"
MODEL_RE = r"[BMT]\d{3}[A-Z]{3}\d{2}\.[A-Z0-9]+"
EMAIL_RE = r"[A-Za-z0-9._%+\-]+@([A-Za-z0-9\-]+(\.[A-Za-z0-9\-]+)*\.[A-Za-z]{2,})"
SENT_MD_RE = r"\*\*Sent:\*\*\s*(.+?)\s*(?=\*\*[A-Za-z]+:\*\*|\n|$)"
SENT_PLAIN_RE = r"(?m)^\s*Sent:\s*(.+?)\s*$"
SENT_EN_FMT = "MMMM d, yyyy h:mm a"
SENT_ZH_RE = r"\d{4}年\d{1,2}月\d{1,2}日"

# §6.3 初表：customer → 比對 regex（主旨＋正文）。只做頻率探勘，正式對照走 entity_alias。
CUSTOMER_PATTERNS = {
    "Acer": r"(?i)\bacer\b",
    "ASUS": r"(?i)\basus\b|華碩",
    "Dell": r"(?i)\bdell\b",
    "HP": r"\bHP\b",
    "Lenovo": r"(?i)\blenovo\b|聯想",
    "MSI": r"\bMSI\b|微星",
    "TCL": r"\bTCL\b",
    "Hisense": r"(?i)\bhisense\b|海信",
    "Samsung VD": r"(?i:\bsamsung\b)|\bVD\b|三星",
    "TPV": r"\bTPV\b|冠捷",
    "SONY": r"(?i)\bsony\b",
}
INTERNAL_DOMAINS = ["auo.com"]

In [ ]:
# [c03] pure_mail_features
def prelim_clean(col: Column) -> Column:
    """粗清洗（只為量長度與 token）：刪圖片佔位、Word 殘渣行、HTML entity，空白壓成一格。"""
    c = F.coalesce(col, F.lit(""))
    c = F.regexp_replace(c, IMG_RE, " ")
    c = F.regexp_replace(c, WORD_JUNK_RE, " ")
    c = F.regexp_replace(c, ENTITY_RE, " ")
    return F.trim(F.regexp_replace(c, r"\s+", " "))


def count_matches(col: Column, pattern: str) -> Column:
    return F.size(F.regexp_extract_all(F.coalesce(col, F.lit("")), F.lit(pattern), F.lit(0)))


def with_mail_features(df: DataFrame) -> DataFrame:
    """mail 層特徵：mail_from 解析、主旨正規化、水位落差、三個正文欄的長度。"""
    mf = F.trim(F.coalesce(F.col("mail_from"), F.lit("")))
    subj = F.coalesce(F.col("mail_subject"), F.lit(""))
    cols = {
        "from_ok": mf.rlike(MAIL_FROM_RE),
        "dept_code": F.nullif(F.regexp_extract(mf, MAIL_FROM_RE, 4), F.lit("")),
        "subject_clean": F.trim(F.regexp_replace(subj, SUBJECT_PREFIX_RE, "")),
        "subject_prefixed": subj.rlike(SUBJECT_PREFIX_RE),
        "unit_tag": F.nullif(F.upper(F.regexp_extract(subj, UNIT_TAG_RE, 1)), F.lit("")),
        "lag_days": F.datediff(F.to_date("create_date"), F.col("mail_date")),
        "attach_count": F.coalesce(F.col("has_attach"), F.lit(0)),
        "top_recipients": (
            F.length(F.trim(F.concat_ws("", "mail_to", "mail_cc", "mail_bcc"))) > 0
        ),
    }
    for f in BODY_FIELDS:
        cols[f"len_{f}"] = F.length(F.coalesce(F.col(f), F.lit("")))
        cols[f"clen_{f}"] = F.length(prelim_clean(F.col(f)))
    df = df.withColumns(cols)
    raw_max = F.greatest(*[F.col(f"len_{f}") for f in BODY_FIELDS])
    return df.withColumn("oversize", raw_max > OVERSIZE_CHARS)

In [ ]:
# [c04] pure_body_choice
def tokens(col: Column) -> Column:
    """粗 token 集合：小寫、非字母數字切開、長度 ≥ 2、去重。中文無空白會成長串，Jaccard 偏低，需搭配客戶集合看。"""
    parts = F.split(F.lower(prelim_clean(col)), r"[^\p{L}\p{N}]+")
    return F.array_distinct(F.filter(parts, lambda x: F.length(x) >= 2))


def customers_in(col: Column) -> Column:
    c = F.coalesce(col, F.lit(""))
    hits = F.array(*[F.when(c.rlike(p), F.lit(n)) for n, p in CUSTOMER_PATTERNS.items()])
    return F.filter(hits, lambda x: x.isNotNull())


def jaccard(a: Column, b: Column) -> Column:
    u = F.size(F.array_union(a, b))
    return F.when((F.size(a) > 0) & (F.size(b) > 0), F.size(F.array_intersect(a, b)) / u)


def with_body_choice(df: DataFrame) -> DataFrame:
    """§4.1 (a) 1–4：選正文來源、算兩兩 Jaccard 與 body_conflict。需先跑 with_mail_features。"""
    src = (
        F.when(F.col("clen_mail_body_md") >= MIN_BODY_CHARS, F.lit("mail_body_md"))
        .when(F.col("clen_mail_without_fw") > 0, F.lit("mail_without_fw"))
        .when(F.col("clen_mail_body") > 0, F.lit("mail_body"))
        .otherwise(F.lit("none"))
    )
    df = df.withColumn("body_source_field", src)
    df = df.withColumn(
        "body_raw",
        F.when(src == "mail_body_md", F.col("mail_body_md"))
        .when(src == "mail_without_fw", F.col("mail_without_fw"))
        .when(src == "mail_body", F.col("mail_body")),
    )
    short = {"mail_body_md": "md", "mail_without_fw": "wofw", "mail_body": "body"}
    df = df.withColumns({f"tok_{short[f]}": tokens(F.col(f)) for f in BODY_FIELDS})
    df = df.withColumns({f"cust_{short[f]}": customers_in(F.col(f)) for f in BODY_FIELDS})
    df = df.withColumns({
        "jac_md_wofw": jaccard(F.col("tok_md"), F.col("tok_wofw")),
        "jac_md_body": jaccard(F.col("tok_md"), F.col("tok_body")),
        "jac_wofw_body": jaccard(F.col("tok_wofw"), F.col("tok_body")),
    })

    def conflict_with(chosen: str, other: str) -> Column:
        a, b = f"tok_{chosen}", f"tok_{other}"
        ca, cb = F.col(f"cust_{chosen}"), F.col(f"cust_{other}")
        return (
            (F.size(F.col(b)) > 0)
            & (jaccard(F.col(a), F.col(b)) < 0.3)
            & (F.size(ca) > 0) & (F.size(cb) > 0)
            & ~F.arrays_overlap(ca, cb)
        )

    conflict = F.lit(False)
    for f, s in short.items():
        others = [o for o in short.values() if o != s]
        conflict = F.when(src == f, conflict_with(s, others[0]) | conflict_with(s, others[1])).otherwise(conflict)
    return df.withColumn("body_conflict", F.coalesce(conflict, F.lit(False))).drop(*[f"tok_{s}" for s in short.values()])

In [ ]:
# [c05] pure_content_type
def with_content_type(df: DataFrame) -> DataFrame:
    """§4.1 (d) 的 mail 層粗估（真正的 T1-b 是 segment 層）。需先跑 with_body_choice。"""
    raw = F.coalesce(F.col("body_raw"), F.lit(""))
    df = df.withColumns({
        **{f"n_{k}": count_matches(raw, p) for k, p in MD_MARKERS.items()},
        "body_clen": F.length(prelim_clean(raw)),
        "clen_noheader": F.length(prelim_clean(F.regexp_replace(raw, HEADER_LINE_RE, " "))),
        "n_models": F.size(F.array_distinct(F.regexp_extract_all(raw, F.lit(MODEL_RE), F.lit(0)))),
        "has_disclaimer": raw.rlike(DISCLAIMER_RE),
        "has_attach_kw": raw.rlike(ATTACH_KW_RE),
        "has_item_header": raw.rlike(ITEM_HEADER_RE),
        "has_minutes_kw": raw.rlike(MINUTES_RE),
    })
    n_from = F.greatest(F.col("n_md_from"), F.col("n_plain_from"))
    clen = F.col("body_clen")
    ctype = (
        F.when(clen < MIN_BODY_CHARS, F.lit("noise"))
        .when((clen < SHORT_CHARS) & ((F.col("attach_count") > 0) | F.col("has_attach_kw")), F.lit("attachment_only"))
        .when((F.col("n_md_when") > 0) & (F.col("clen_noheader") < SHORT_CHARS), F.lit("appointment"))
        .when(
            F.col("has_item_header") | (F.col("n_models") >= 3)
            | (F.col("has_minutes_kw") & (clen >= MINUTES_CHARS)),
            F.lit("inline_minutes"),
        )
        .when((n_from >= 2) & (clen >= SHORT_CHARS), F.lit("thread_reply"))
        .otherwise(F.lit("unclassified"))
    )
    est_seg = F.greatest(n_from, F.col("n_orig_sep")) + F.lit(1)
    return df.withColumns({"content_type_est": ctype, "n_from_est": n_from, "est_segments": est_seg})

In [ ]:
# [c06] pure_explodes
def explode_sent_values(df: DataFrame) -> DataFrame:
    """每封信的 Sent: 值（md 與純文字兩種標頭）→ (pk_seq, subject_clean, sent_raw, sent_ts, sent_kind)。"""
    raw = F.coalesce(F.col("body_raw"), F.lit(""))
    vals = F.concat(
        F.regexp_extract_all(raw, F.lit(SENT_MD_RE), F.lit(1)),
        F.regexp_extract_all(raw, F.lit(SENT_PLAIN_RE), F.lit(1)),
    )
    out = df.select("pk_seq", "subject_clean", F.explode(vals).alias("sent_raw"))
    # Java 的 \s 不含 U+00A0（不換行空白），trim 也去不掉，要另外列
    s = F.trim(F.regexp_replace(F.regexp_replace("sent_raw", ENTITY_RE, " "), r"[\s\u00A0]+", " "))
    s = F.regexp_replace(s, r"^[A-Za-z]+,\s*", "")  # 去星期
    ts = F.try_to_timestamp(s, F.lit(SENT_EN_FMT))
    kind = (
        F.when(ts.isNotNull(), F.lit("en"))
        .when(F.col("sent_raw").rlike(SENT_ZH_RE), F.lit("zh"))
        .otherwise(F.lit("unparsed"))
    )
    return out.withColumns({"sent_norm": s, "sent_ts": ts, "sent_kind": kind})


def segment_repeat_stats(df_sent: DataFrame) -> DataFrame:
    """跨信重複粗估：同主旨＋同 Sent 時間的段出現在幾封信。repeat_rate = 重複段數 / 段實例數。"""
    per = (
        df_sent.select("pk_seq", "subject_clean", "sent_norm").distinct()
        .groupBy("subject_clean", "sent_norm").agg(F.count("*").alias("n_pk"))
    )
    return per.agg(
        F.sum("n_pk").alias("segment_instances"),
        F.count("*").alias("distinct_segments"),
        F.sum(F.col("n_pk") - 1).alias("repeat_instances"),
        F.max("n_pk").alias("max_copies"),
    ).withColumn("repeat_rate", F.round(F.col("repeat_instances") / F.col("segment_instances"), 3))


def explode_email_domains(df: DataFrame) -> DataFrame:
    """正文＋頂層收件欄中的 email → (pk_seq, domain)，一封信同網域只算一次。不保留 email 本身。"""
    text = F.concat_ws(" ", "body_raw", "mail_to", "mail_cc", "mail_bcc")
    doms = F.regexp_extract_all(text, F.lit(EMAIL_RE), F.lit(1))
    return (
        df.select("pk_seq", F.explode(F.array_distinct(F.transform(doms, F.lower))).alias("domain"))
        .withColumn("is_internal", F.col("domain").isin(INTERNAL_DOMAINS))
    )


def explode_customers(df: DataFrame) -> DataFrame:
    """主旨＋正文命中的客戶 → (pk_seq, customer, in_subject)。"""
    text = F.concat_ws(" ", "mail_subject", "body_raw")
    subj_hits = customers_in(F.col("mail_subject"))
    return (
        df.select("pk_seq", subj_hits.alias("_subj"), F.explode(customers_in(text)).alias("customer"))
        .withColumn("in_subject", F.array_contains("_subj", F.col("customer")))
        .drop("_subj")
    )


def num_profile(df: DataFrame, cols: list[str]) -> DataFrame:
    """數值欄的 P50 / P95 / max，一列。"""
    aggs = []
    for c in cols:
        aggs += [
            F.percentile_approx(c, 0.5).alias(f"{c}_p50"),
            F.percentile_approx(c, 0.95).alias(f"{c}_p95"),
            F.max(c).alias(f"{c}_max"),
        ]
    return df.agg(*aggs)

In [ ]:
# [c06a] pure_md_bloat
BLOAT_PATTERNS = {
    "data_uri": r"data:[a-z]+/[a-z0-9.+-]+;base64,",
    "base64_run": r"[A-Za-z0-9+/=]{200,}",
    "html_tag": r"<[A-Za-z/!][^>\n]{0,300}>",
    "html_entity": r"&[A-Za-z]+;|&#\d+;",
    "md_table_line": r"(?m)^\s*\|.*\|\s*$",
    "md_image": r"!\[[^\]]*\]\([^)]*\)",
    "md_link": r"(?<!!)\[[^\]]*\]\([^)]*\)",
    "url": r"https?://\S+",
}


def md_bloat_profile(df: DataFrame, col: str = "mail_body_md") -> DataFrame:
    """md 正文的組成：各樣式出現次數、佔用字數與佔比（只有數字，不含正文）。樣式之間可能重疊，佔比加總可 > 1。"""
    c = F.coalesce(F.col(col), F.lit(""))
    total = F.greatest(F.length(c), F.lit(1))
    lines = F.split(c, "\n")
    cols = {
        "total_chars": F.length(c),
        "n_lines": F.size(lines),
        "n_blank_lines": F.size(F.filter(lines, lambda x: F.trim(x) == "")),
        "max_line_chars": F.array_max(F.transform(lines, F.length)),
        "share_whitespace": F.round((F.length(c) - F.length(F.regexp_replace(c, r"\s", ""))) / total, 3),
    }
    for k, p in BLOAT_PATTERNS.items():
        hits = F.regexp_extract_all(c, F.lit(p), F.lit(0))
        cols[f"n_{k}"] = F.size(hits)
        cols[f"share_{k}"] = F.round(F.aggregate(hits, F.lit(0), lambda acc, x: acc + F.length(x)) / total, 3)
    return df.select("pk_seq", "len_mail_without_fw", "len_mail_body", *[v.alias(k) for k, v in cols.items()])


In [ ]:
# [c10] load_source
# 先過濾日期（mail_date 窗）再算特徵；不讀 mail_body_html。
report = {"cfg": {k: str(v) for k, v in cfg.items()}}


def one(df: DataFrame) -> dict:
    """單列彙總 → dict。"""
    r = df.limit(1).collect()
    return r[0].asDict() if r else {}


def top(df: DataFrame, n: int = 30) -> list[dict]:
    return [r.asDict() for r in df.limit(n).collect()]


def show(title: str, obj) -> None:
    print(f"\n== {title}")
    if isinstance(obj, list):
        for r in obj:
            print("  ", r)
    else:
        print("  ", obj)


src_fq = f"{cfg['catalog']}.{cfg['schema']}.{cfg['source_table']}"
df_window = (
    spark.table(src_fq)
    .where(F.col("mail_date").between(F.lit(cfg["date_from"]), F.lit(cfg["date_to"])))
    .select(*SOURCE_COLS)
)
df_feat = with_content_type(with_body_choice(with_mail_features(df_window)))
df_mail = df_feat.where(~F.coalesce(F.col("is_test"), F.lit(False)))
print(src_fq, cfg["date_from"], "~", cfg["date_to"])

In [ ]:
# [c11] stat_basic
r = one(df_feat.agg(
    F.count("*").alias("rows_in_window"),
    F.sum(F.coalesce(F.col("is_test"), F.lit(False)).cast("int")).alias("is_test_rows"),
    F.min("create_date").alias("create_min"), F.max("create_date").alias("create_max"),
    F.min("mail_date").alias("mail_min"), F.max("mail_date").alias("mail_max"),
    F.sum((F.col("lag_days") > 7).cast("int")).alias("lag_gt_7d"),
    F.sum((F.col("lag_days") < 0).cast("int")).alias("lag_negative"),
    F.sum(F.col("top_recipients").cast("int")).alias("top_recipients_nonempty"),
    F.sum(F.coalesce(F.col("is_fw"), F.lit(False)).cast("int")).alias("is_fw_rows"),
    F.sum((F.col("attach_count") > 0).cast("int")).alias("has_attach_rows"),
))
r.update(one(num_profile(df_mail, ["lag_days"])))
by_month = top(df_mail.groupBy(F.date_format("mail_date", "yyyy-MM").alias("m")).count().orderBy("m"), 24)
report["basic"] = r
report["mails_by_month"] = by_month
show("basic", r)
show("mails_by_month（排除 is_test）", by_month)

In [ ]:
# [c12] stat_body_fields
# §4.0 關卡：mail_body_md 非空 < 80% 先停。clen = 粗清洗後長度。
aggs = [F.count("*").alias("n")]
for f in BODY_FIELDS:
    aggs += [
        F.round(F.avg((F.col(f"clen_{f}") > 0).cast("int")), 3).alias(f"nonempty_{f}"),
        F.round(F.avg((F.col(f"clen_{f}") >= MIN_BODY_CHARS).cast("int")), 3).alias(f"ge150_{f}"),
    ]
aggs += [F.sum(F.col("oversize").cast("int")).alias("oversize_rows")]
r = one(df_mail.agg(*aggs))
r.update(one(num_profile(df_mail, [f"clen_{f}" for f in BODY_FIELDS] + ["len_mail_body_md"])))
src_dist = top(df_mail.groupBy("body_source_field").count().orderBy(F.desc("count")))
report["body_fields"] = r
report["body_source_dist"] = src_dist
show("body_fields", r)
show("body_source_field 分布（§4.1 (a) 規則）", src_dist)

In [ ]:
# [c13] stat_mail_from_unit
# 只印部門代碼（AUO 內部）與計數，不印姓名 / 帳號。
r = one(df_mail.agg(
    F.round(F.avg(F.col("from_ok").cast("int")), 3).alias("from_format_ok_rate"),
    F.countDistinct("dept_code").alias("distinct_dept_codes"),
    F.round(F.avg(F.col("unit_tag").isNotNull().cast("int")), 3).alias("unit_tag_rate"),
    F.round(F.avg(F.col("subject_prefixed").cast("int")), 3).alias("subject_fw_re_rate"),
))
dept = top(df_mail.groupBy("dept_code").count().orderBy(F.desc("count")), 50)
prefix_x_tag = top(
    df_mail.groupBy(F.substring("dept_code", 1, 4).alias("dept_prefix4"), "unit_tag")
    .count().orderBy(F.desc("count")), 50,
)
report["mail_from"] = r
report["dept_codes"] = dept
report["dept_prefix_x_unit_tag"] = prefix_x_tag
show("mail_from / 主旨", r)
show("dept_code（待決 #13）", dept)
show("dept 前 4 碼 × 主旨單位標頭（§6.8 衝突率）", prefix_x_tag)

In [ ]:
# [c14] stat_md_markers_and_sent
# md 標頭覆蓋率（依選用的正文來源分組）與 Sent: 日期可解析率（§4.1 (c)）。
marker_cols = [f"n_{k}" for k in MD_MARKERS]
markers = top(
    df_mail.groupBy("body_source_field").agg(
        F.count("*").alias("n"),
        *[F.round(F.avg((F.col(c) > 0).cast("int")), 3).alias(f"has_{c[2:]}") for c in marker_cols],
        F.round(F.avg(F.col("has_disclaimer").cast("int")), 3).alias("has_disclaimer"),
    ).orderBy(F.desc("n"))
)
df_sent = explode_sent_values(df_mail)
sent = top(df_sent.groupBy("sent_kind").count().orderBy(F.desc("count")))
unparsed = top(
    df_sent.where(F.col("sent_kind") == "unparsed")
    .select(F.substring("sent_norm", 1, 40).alias("v")).groupBy("v").count().orderBy(F.desc("count")),
    10,
)
report["md_markers"] = markers
report["sent_parse"] = sent
report["sent_unparsed_top"] = unparsed
show("標頭標記覆蓋率", markers)
show("Sent: 解析（en = 英文長日期可解析）", sent)
show("Sent: 解析失敗值 top 10", unparsed)

In [ ]:
# [c15] stat_consistency
# 待決 #1：三正文是否講同一件事。body_conflict > 10% 先停（§4.0b 關卡）。
r = one(df_mail.agg(
    F.round(F.avg(F.col("body_conflict").cast("int")), 3).alias("body_conflict_rate"),
    F.sum(F.col("body_conflict").cast("int")).alias("body_conflict_rows"),
    *[F.round(F.avg(c), 3).alias(f"{c}_avg") for c in ["jac_md_wofw", "jac_md_body", "jac_wofw_body"]],
    *[F.round(F.avg((F.col(c) < 0.3).cast("int")), 3).alias(f"{c}_lt03")
      for c in ["jac_md_wofw", "jac_md_body", "jac_wofw_body"]],
))
conflict_pk = [x["pk_seq"] for x in top(df_mail.where("body_conflict").select("pk_seq").orderBy("pk_seq"), 20)]
report["consistency"] = r
report["body_conflict_pk_sample"] = conflict_pk
show("一致性（Jaccard 為粗 token；_lt03 = 低於 0.3 的比例）", r)
show("body_conflict 的 pk_seq（最多 20 筆，可用 meeting_record_explore 看原文）", conflict_pk)

In [ ]:
# [c16] stat_content_type
ctype = top(
    df_mail.groupBy("content_type_est").agg(
        F.count("*").alias("n"),
        F.percentile_approx("body_clen", 0.5).alias("clen_p50"),
        F.sum("est_segments").alias("est_segments"),
    ).orderBy(F.desc("n"))
)
r = one(num_profile(df_mail, ["est_segments", "n_models", "body_clen"]))
r.update(one(df_mail.agg(
    F.sum((F.col("body_clen") > 12_000).cast("int")).alias("body_gt_12k"),  # §4.1 (c) sub_no 門檻
    F.sum("est_segments").alias("est_segments_total"),
)))
report["content_type_est"] = ctype
report["segments"] = r
show("content_type 粗估（mail 層）", ctype)
show("每封段數 / 型號數 / 長度", r)

In [ ]:
# [c17] stat_domains_customers
# 只列網域（不列 email）；§6.3 entity_alias 的 email_domain 種子與 side=unknown 比例用。
df_dom = explode_email_domains(df_mail)
domains = top(df_dom.groupBy("domain", "is_internal").agg(F.countDistinct("pk_seq").alias("mails"))
              .orderBy(F.desc("mails")), 60)
df_cust = explode_customers(df_mail)
customers = top(
    df_cust.groupBy("customer").agg(
        F.countDistinct("pk_seq").alias("mails"),
        F.sum(F.col("in_subject").cast("int")).alias("in_subject"),
    ).orderBy(F.desc("mails"))
)
r = one(df_mail.join(df_cust.select("pk_seq").distinct().withColumn("_hit", F.lit(1)), "pk_seq", "left")
        .agg(F.round(F.avg(F.col("_hit").isNull().cast("int")), 3).alias("no_customer_rate")))
report["email_domains"] = domains
report["customers"] = customers
report["customer_cover"] = r
show("email 網域（每網域出現在幾封信）", domains)
show("客戶頻率（§6.3 初表 regex）", customers)
show("找不到任何客戶的信比例", r)

In [ ]:
# [c18] stat_duplicates
# §4.1 (e)：同主旨＋同 Sent 時間的段出現在多封信 → 轉寄重複的量級。另看整封正文完全相同。
seg = one(segment_repeat_stats(df_sent))
exact = one(
    df_mail.where(F.col("body_clen") >= MIN_BODY_CHARS)
    .groupBy(F.sha1(prelim_clean(F.col("body_raw"))).alias("h")).count()
    .agg(F.sum(F.when(F.col("count") > 1, F.col("count") - 1).otherwise(0)).alias("exact_dup_mails"))
)
subj = one(
    df_mail.groupBy("subject_clean").count()
    .agg(F.count("*").alias("distinct_subjects"), F.max("count").alias("max_mails_per_subject"),
         F.sum(F.when(F.col("count") > 1, F.col("count")).otherwise(0)).alias("mails_in_repeated_subjects"))
)
report["dup_segments"] = seg
report["dup_exact"] = exact
report["dup_subjects"] = subj
show("段重複（repeat_rate 預期 0.3–0.6）", seg)
show("整封正文完全相同", exact)
show("主旨重複", subj)

In [ ]:
# [c19] stat_md_bloat
# md 正文為何比其他正文大很多：最長的 8 封拆成各樣式佔比（只有數字）。
bloat = top(md_bloat_profile(df_mail.where(F.col("len_mail_body_md") > 0)).orderBy(F.desc("total_chars")), 8)
report["md_bloat_top"] = bloat
show("md 最長的 8 封：各樣式佔比（share_* = 佔總字數比例）", bloat)


In [ ]:
# [c20] stat_sales_brief
# 待決 #10：文字端欄位；待決 #3：填報時間落在幾號（判月份歸屬截止日）。
from pyspark.sql.types import DateType, TimestampType

sb = {}
for t in ["g_sales_brief_brand_summary", "g_sales_brief_app_summary", "g_sales_brief_market_intelligence"]:
    fq = f"{cfg['catalog']}.{cfg['schema']}.{t}"
    if not spark.catalog.tableExists(fq):
        sb[t] = {"exists": False}
        show(t, "不存在")
        continue
    d = spark.table(fq)
    info = {"exists": True, "columns": [f"{f.name}:{f.dataType.simpleString()}" for f in d.schema.fields]}
    if {"app_code", "fill_year", "fill_month"} <= set(d.columns):
        per = d.withColumn("period", F.col("fill_year") * 100 + F.col("fill_month"))
        latest = [x["period"] for x in top(per.select("period").distinct().orderBy(F.desc("period")), cfg["sb_periods"])]
        per = per.where(F.col("period").isin(latest))
        ts_cols = [f.name for f in d.schema.fields if isinstance(f.dataType, (TimestampType, DateType))]
        info["periods"] = top(
            per.groupBy("app_code", "period").agg(
                F.count("*").alias("rows"),
                *[F.max(c).alias(f"max_{c}") for c in ts_cols],
                *[F.min(c).alias(f"min_{c}") for c in ts_cols],
            ).orderBy("app_code", "period"), 60,
        )
        if "brand_code" in d.columns:
            info["brands"] = top(
                per.groupBy("app_code").agg(F.sort_array(F.collect_set("brand_code")).alias("brand_codes"))
                .orderBy("app_code")
            )
        if "field_key" in d.columns:
            info["field_keys"] = top(per.groupBy("field_key").count().orderBy(F.desc("count")), 40)
    sb[t] = info
    show(t, info)
report["sales_brief"] = sb

In [ ]:
# [c21] stat_semantic_tables
# 待決 #15：entity_alias 是否已建；待決 #13：有無部門代碼對照表可用。
sem = {}
for fq in [f"{cfg['catalog']}.mi3_semantic.entity_alias", f"{cfg['catalog']}.mi3_semantic.dim_entity"]:
    ok = spark.catalog.tableExists(fq)
    sem[fq] = {"exists": ok}
    if ok:
        d = spark.table(fq)
        sem[fq]["columns"] = [f"{f.name}:{f.dataType.simpleString()}" for f in d.schema.fields]
        if "alias_type" in d.columns:
            sem[fq]["alias_types"] = top(d.groupBy("alias_type").count())
cands = top(spark.sql(f"""
    SELECT table_schema, table_name
    FROM {cfg['catalog']}.information_schema.tables
    WHERE lower(table_name) RLIKE 'dept|department|org_|organization|employee|emp_|hr_'
    ORDER BY table_schema, table_name
"""), 50)
report["semantic"] = sem
report["dept_table_candidates"] = cands
show("mi3_semantic", sem)
show("部門 / 人員對照表候選", cands)

In [ ]:
# [c30] report_json
# 整段複製貼回給 Agent。只有統計、網域、AUO 部門代碼、主旨無關的值；不含姓名與 email。
print(json.dumps(report, ensure_ascii=False, default=str, separators=(",", ":")))